# Trees and XGBoost classifiers — iris

**Kya chahiye:** Google Colab ya local Python 3.10+. `Run all` karo; first cell missing packages install karta hai.

**Dataset:** scikit-learn ka canonical `load_iris` (150 rows, 4 features, 3 species). Ye chhota dataset sirf concepts ke liye hai — numbers ko production evidence mat samjho.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "sklearn": "scikit-learn==1.5.2",
    "xgboost": "xgboost==2.1.3",
    "pandas": "pandas==2.2.3",
}
missing = [pip_name for module, pip_name in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import sklearn
import xgboost
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("Runtime is ready. This pack is for Colab or a local environment, not the offline Code Bro kernel.")

## 1. Load, split and keep the class names

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

data = load_iris(as_frame=True)
df = data.frame.rename(columns={"target": "species"})
X = df.drop(columns="species")
y = df["species"]
class_names = list(data.target_names)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("shape:", df.shape, "| classes:", class_names)
print("train:", X_train.shape, "| test:", X_test.shape)

## 2. Three models, one comparison table

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

models = {
    "decision_tree": DecisionTreeClassifier(max_depth=3, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1),
    "xgboost": XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9, objective="multi:softprob",
        eval_metric="mlogloss", random_state=42, n_jobs=4,
    ),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"{name:>16}: accuracy={accuracy_score(y_test, pred):.3f}")
    print(classification_report(y_test, pred, target_names=class_names, digits=3))

## 3. Tree depth is a capacity decision

In [ ]:
for depth in range(1, 7):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"depth={depth}: train={train_acc:.3f} | test={test_acc:.3f}")

## 4. The confusion matrix shows which species get mixed

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

pred = models["xgboost"].predict(X_test)
matrix = confusion_matrix(y_test, pred)
print(pd.DataFrame(matrix, index=[f"actual {n}" for n in class_names], columns=[f"pred {n}" for n in class_names]))
print("rows sum to test size:", matrix.sum() == len(y_test))

## 5. Limitations

- Iris is tiny, balanced and easy. A high score here does not transfer to a messy business dataset.
- A single train/test split can be lucky or unlucky. Use cross-validation before choosing a model.
- Boosted trees can overfit small data; depth, learning rate and tree count are capacity controls.
- Feature importance describes this fitted model, not the biological importance of a measurement.
- The offline Code Bro kernel intentionally does not ship scikit-learn or XGBoost — run this pack in Colab or a local environment.

**Exercise:** replace the 25% holdout with 5-fold stratified cross-validation and compare mean accuracy and standard deviation for all three models.